# Pharmaceutical R&D Document Research & Critique
Agentic deep research and critique of pharmaceutical documents using Dataiku LLM Mesh + SerpAPI.

In [ ]:
# Run once to install dependencies (comment out after first run)
# %pip install -r ../requirements.txt

In [ ]:
import sys
import os

sys.path.insert(0, "..")

from IPython.display import display, Markdown, clear_output

from src import ResearchCritiqueSystem, Config
from src.llm.mesh_client import MeshClient

In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIGURE HERE — edit these variables then run this cell
# ═══════════════════════════════════════════════════════

# Dataiku managed folder containing the documents
FOLDER_NAME = "research_document"

# Paste one of the LLM IDs printed above
LLM_ID = available_llms[0] if available_llms else "openai:gpt-4o"

# SerpAPI key — pre-filled from project variables, override if needed
SERP_API_KEY = _serp_key_from_vars or "YOUR_SERP_API_KEY"

# Describe the analysis you want
ANALYSIS_DESCRIPTION = (
    "Evaluate this document for scientific rigor, regulatory alignment "
    "with FDA/EMA guidelines, and identify evidence gaps."
)

# Agent tuning
MAX_AGENT_STEPS = 12
MIN_WEB_SEARCHES = 3

# ── List files in folder ───────────────────────────────
import dataiku
_folder = dataiku.Folder(FOLDER_NAME)
_folder_files = _folder.list_paths_in_partition()

if _folder_files:
    print(f"Files in '{FOLDER_NAME}':")
    for f in _folder_files:
        print(f"  {f}")
else:
    print(f"⚠ No files found in '{FOLDER_NAME}'. Upload documents there first.")

# Set to one of the filenames printed above
DOCUMENT_NAME = _folder_files[0] if _folder_files else ""

# ── Validation ─────────────────────────────────────────
_ok = True
if not SERP_API_KEY or SERP_API_KEY == "YOUR_SERP_API_KEY":
    print("⚠ Set SERP_API_KEY above.")
    _ok = False
if not DOCUMENT_NAME:
    print("⚠ Set DOCUMENT_NAME above.")
    _ok = False
if not ANALYSIS_DESCRIPTION:
    print("⚠ Set ANALYSIS_DESCRIPTION above.")
    _ok = False
if _ok:
    print(f"\n✓ Folder:   {FOLDER_NAME}")
    print(f"✓ LLM:      {LLM_ID}")
    print(f"✓ Document: {DOCUMENT_NAME}")
    print("✓ Ready — run the next cell to start.")

In [ ]:
# ── Run analysis ───────────────────────────────────────────────────────────
import tempfile, pathlib

# Download the selected file from the Dataiku folder to a local temp file
_suffix = pathlib.Path(DOCUMENT_NAME).suffix or ".pdf"
_tmp = tempfile.NamedTemporaryFile(delete=False, suffix=_suffix)
with _folder.get_download_stream(DOCUMENT_NAME) as _stream:
    _tmp.write(_stream.read())
_tmp.close()
DOCUMENT_PATH = _tmp.name
print(f"Downloaded: {DOCUMENT_NAME}")

config = Config(
    llm_id=LLM_ID,
    serp_api_key=SERP_API_KEY,
    max_iterations=MAX_AGENT_STEPS,
    min_searches_required=MIN_WEB_SEARCHES,
)

system = ResearchCritiqueSystem(config)
accumulated = []

for chunk in system.run_stream(file_path=DOCUMENT_PATH, description=ANALYSIS_DESCRIPTION):
    accumulated.append(chunk)
    clear_output(wait=True)
    display(Markdown("".join(accumulated)))

In [ ]:
# ── Run analysis ───────────────────────────────────────────────────────────

config = Config(
    llm_id=LLM_ID,
    serp_api_key=SERP_API_KEY,
    max_iterations=MAX_AGENT_STEPS,
    min_searches_required=MIN_WEB_SEARCHES,
)

system = ResearchCritiqueSystem(config)
accumulated = []

for chunk in system.run_stream(file_path=DOCUMENT_PATH, description=ANALYSIS_DESCRIPTION):
    accumulated.append(chunk)
    clear_output(wait=True)
    display(Markdown("".join(accumulated)))